# DeepSORVF — Colab Setup & Ablation Study

**Structure:**
```
DeepSORVF_Project/
├── data/clips/        ← video data (clip-01, Video-29, ...)
├── config/            ← YAML configs
├── utils/             ← source code
├── detection_yolox/   ← YOLOX detector
├── detection_yolov8/  ← YOLOv8 maritime detector
├── deep_sort/         ← DeepSORT tracker
├── weights/           ← model weights
├── run_ablation.py    ← headless runner
└── ablation_runner.py ← batch runner
```

**Phases:**
- Phase 1: C0, C1, C2 (no AIS)
- Phase 2: C3, C4, C5 (with AIS)

If Phase 2 stops, restart it without re-running Phase 1.

In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

ZIP_PATH = '/content/drive/MyDrive/DeepSORVF_Colab.zip'
PROJECT_ROOT = '/content/drive/MyDrive/DeepSORVF_Project'
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# Step 2: Extract zip and clear Python cache
import os, zipfile, shutil, glob

if os.path.exists(ZIP_PATH):
    print('Extracting zip (overwriting)...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(PROJECT_ROOT)
    print(f'Extracted to {PROJECT_ROOT}')
else:
    print(f'ERROR: {ZIP_PATH} not found!')

# Clear all __pycache__
for pycache in glob.glob(os.path.join(PROJECT_ROOT, '**', '__pycache__'), recursive=True):
    shutil.rmtree(pycache)
print('Cleared __pycache__ directories.')

In [ ]:
# Step 3: Install dependencies
!pip install ultralytics==8.4.121
!pip install filterpy lap easydict geopy pyproj fastdtw loguru
!pip install scikit-image

import torch
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')

In [ ]:
# Step 4: Verify weights
weights_dir = os.path.join(PROJECT_ROOT, 'weights')
required = {
    'best.pt': 40_000_000,
    'YOLOX-final.pth': 30_000_000,
    'ckpt.t7': 40_000_000,
}

all_ok = True
for name, min_size in required.items():
    path = os.path.join(weights_dir, name)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1_000_000
        status = 'OK' if size_mb > min_size / 1_000_000 else 'too small'
        print(f'  {name}: {size_mb:.1f} MB {status}')
    else:
        print(f'  {name}: MISSING')
        all_ok = False

print(f'\nWeights: {"OK" if all_ok else "MISSING FILES"}')

In [ ]:
# Step 5: Verify clips
clips_dir = os.path.join(PROJECT_ROOT, 'data', 'clips')
clips = ['clip-01', 'clip-02', 'clip-10', 'Video-10', 'Video-28', 'Video-29', 'Video-34']

for clip in clips:
    clip_path = os.path.join(clips_dir, clip)
    if os.path.isdir(clip_path):
        files = os.listdir(clip_path)
        video = [f for f in files if f.endswith('.mp4') or f.endswith('.avi')]
        ais_dir = os.path.join(clip_path, 'ais')
        ais_count = len(os.listdir(ais_dir)) if os.path.isdir(ais_dir) else 0
        print(f'  {clip}: video={video[0] if video else "MISSING"}, ais={ais_count}')
    else:
        print(f'  {clip}: NOT FOUND')

print('\nDone.')

In [ ]:
# Step 6: Quick test — load models
import sys
sys.path.insert(0, PROJECT_ROOT)

from detection_yolox.yolo import YOLO
yolo = YOLO()
print('YOLOX loaded')

from detection_yolov8.yolov8_detector import YOLOv8Detector, ULTRALYTICS_OK
if ULTRALYTICS_OK:
    yolov8 = YOLOv8Detector(weights=os.path.join(PROJECT_ROOT, 'weights/best.pt'), conf=0.30)
    print('YOLOv8 loaded')

---
## Phase 1: C0, C1, C2 (no AIS)
Configurations without AIS fusion. Safe to restart if interrupted.

In [ ]:
# Phase 1: Run C0, C1, C2 on all clips
os.chdir(PROJECT_ROOT)

from run_ablation import run_pipeline
import json

CLIPS = ['clip-01', 'clip-02', 'clip-10', 'Video-10', 'Video-28', 'Video-29', 'Video-34']
PHASE1_CONFIGS = {
    'C0': dict(use_ensemble=False, use_static_filter=False, ais_enabled=False, anti=0),
    'C1': dict(use_ensemble=True,  use_static_filter=False, ais_enabled=False, anti=0),
    'C2': dict(use_ensemble=True,  use_static_filter=True,  ais_enabled=False, anti=0),
}

RESULT_DIR = os.path.join(PROJECT_ROOT, 'ablation_results', 'phase1')
os.makedirs(RESULT_DIR, exist_ok=True)

phase1_results = []
for clip in CLIPS:
    for config_name, flags in PHASE1_CONFIGS.items():
        res_dir = os.path.join(RESULT_DIR, clip, config_name)
        print(f'\n--- {config_name} / {clip} ---')
        try:
            stats = run_pipeline(
                clip_name=clip,
                result_dir=res_dir,
                config_name=config_name,
                **flags
            )
            phase1_results.append(stats)
        except Exception as e:
            print(f'  ERROR: {e}')
            phase1_results.append({'config': config_name, 'clip': clip, 'error': str(e)})

# Save Phase 1 results
with open(os.path.join(RESULT_DIR, 'phase1_summary.json'), 'w') as f:
    json.dump(phase1_results, f, indent=2)

print(f'\n=== Phase 1 complete: {len(phase1_results)} runs ===')

---
## Phase 2: C3, C4, C5 (with AIS)
Configurations with AIS fusion. Can restart independently — Phase 1 results are saved.

In [ ]:
# Phase 2: Run C3, C4, C5 on all clips
os.chdir(PROJECT_ROOT)

from run_ablation import run_pipeline
import json

CLIPS = ['clip-01', 'clip-02', 'clip-10', 'Video-10', 'Video-28', 'Video-29', 'Video-34']
PHASE2_CONFIGS = {
    'C3': dict(use_ensemble=True, use_static_filter=True, ais_enabled=True, anti=0),
    'C4': dict(use_ensemble=True, use_static_filter=True, ais_enabled=True, anti=1),
    'C5': dict(use_ensemble=True, use_static_filter=True, ais_enabled=True, anti=1),
}

RESULT_DIR = os.path.join(PROJECT_ROOT, 'ablation_results', 'phase2')
os.makedirs(RESULT_DIR, exist_ok=True)

phase2_results = []
for clip in CLIPS:
    for config_name, flags in PHASE2_CONFIGS.items():
        res_dir = os.path.join(RESULT_DIR, clip, config_name)
        print(f'\n--- {config_name} / {clip} ---')
        try:
            stats = run_pipeline(
                clip_name=clip,
                result_dir=res_dir,
                config_name=config_name,
                **flags
            )
            phase2_results.append(stats)
        except Exception as e:
            print(f'  ERROR: {e}')
            phase2_results.append({'config': config_name, 'clip': clip, 'error': str(e)})

# Save Phase 2 results
with open(os.path.join(RESULT_DIR, 'phase2_summary.json'), 'w') as f:
    json.dump(phase2_results, f, indent=2)

print(f'\n=== Phase 2 complete: {len(phase2_results)} runs ===')

---
## Results Summary

In [ ]:
# Combine Phase 1 + Phase 2 results
import json, os

RESULT_DIR = os.path.join(PROJECT_ROOT, 'ablation_results')
all_results = []

for phase in ['phase1', 'phase2']:
    summary_file = os.path.join(RESULT_DIR, phase, f'{phase}_summary.json')
    if os.path.exists(summary_file):
        with open(summary_file) as f:
            all_results.extend(json.load(f))

# Print summary table
print(f"{'Config':<6} {'Clip':<12} {'Frames':>8} {'Det-Sec':>8} {'Time':>8} {'ms/frm':>8}")
print('-' * 60)
for r in all_results:
    if 'error' in r:
        print(f"{r['config']:<6} {r['clip']:<12} ERROR: {r['error']}")
    else:
        print(f"{r['config']:<6} {r['clip']:<12} {r['total_frames']:>8} {r['detection_seconds']:>8} {r['wall_time_s']:>7.1f}s {r['avg_ms_per_frame']:>7.1f}")

print(f'\nTotal runs: {len(all_results)}')